# Delay 统计面板 Demo（v0.5.0：象限布局 + 自定义状态图）

统计面板**默认开启**（桶宽 100，无需任何 stats 参数）：在**明细窗右侧**显示——

- 三幅竖直柱状图按“象限”布局：**成功率**（左上，默认按 `state == "PASS"`）、
  **HANG率**（左下，默认 `state == "HANG"`）、**自定义状态图**（右侧整列，
  本例 `stats_custom="FAIL"`，柱色 `#18097d`）；
- 横轴按 delay 时间轴左→右递增（桶标签如 `500-600`、`600-700`…），柱从基线向上生长，异常点对照邻桶一眼可见；
- 悬停柱条可见精确值（原生 tooltip：`2700-2800 · HANG 5/6 · 83.3%`）；
- 数据源是 CSV 本尊：滚出显示窗口的行也计入统计；
- 前端节点池恒定 + 统计按 `stats_interval=` 节流推送，不引入卡顿。


In [1]:
import time, random
import viewviewview as faultviz
assert faultviz.__version__ == "0.5.0", faultviz.__version__
COLUMNS = {"state": str, "delay": int, "pulse": float, "errcode": int}
vt = faultviz.ViewWidget(
    columns=COLUMNS,     # 统计面板默认开启：桶宽 100（0=关闭，可改单位）
    stats_custom="FAIL", # 自定义第三状态图（右侧整列；不填=不显示）
    flush_every=100,     # 弱机/Defender 环境建议；默认是每行刷盘
)
print('widget ready:', vt.csv_path)


widget ready: D:\project\ViewViewView\dev\20261110-172932.csv


In [2]:
# replace=True：重跑本 cell 会先关闭上一个面板（防累积）
vt.show(replace=True)


2026-11-10 17:29:32,731 - viewviewview - INFO - No data recorded yet; showing the declared columns.


## 注入

先突发 150 条，再按 ~100 条/秒稳速注入 600 条（约 6 s）：
- 低延迟段（< 700）大概率 PASS；
- 高延迟段（≥ 2000，即 2700-2800 桶附近）大概率 HANG——该桶在时间轴最右端，异常柱一眼可见。


In [3]:
def inject(long_tail: bool) -> None:
    if long_tail:
        delay = random.randint(2400, 2800)
    elif random.random() < 0.08:
        delay = random.randint(1500, 2300)
    else:
        delay = int(abs(random.gauss(240, 160)))
    if delay >= 2000:
        state = "HANG" if random.random() < 0.75 else "FAIL"
    elif delay >= 700:
        state = "FAIL" if random.random() < 0.5 else "PASS"
    else:
        state = "PASS" if random.random() < 0.94 else "FAIL"
    reply = "ok" if state == "PASS" else ("no-reply" if state == "HANG" else "short")
    vt.update(
        state=state,
        delay=delay,
        pulse=10 + (delay % 40) + 0.5,
        errcode=0 if state == "PASS" else random.randint(1, 8),
    )

start = time.perf_counter()
for _ in range(150):
    inject(False)
print(f'burst 150 rows in {time.perf_counter() - start:.2f}s; total={vt.rows_written}')


burst 150 rows in 0.00s; total=150


In [4]:
start = time.perf_counter()
for i in range(600):
    inject(long_tail=(i % 37 == 0))
    time.sleep(0.01)          # ~100 条/秒
print(f'steady 600 rows in {time.perf_counter() - start:.1f}s; total={vt.rows_written}')
print('观察右侧：横轴按 delay 时间轴递增，HANG率 图右端 2700-2800 桶仍是高峰')


steady 600 rows in 7.2s; total=750
观察右侧：横轴按 delay 时间轴递增，HANG率 图右端 2700-2800 桶仍是高峰


## 想改单位？（如 2700-2800 → 每 10 一桶）

统计参数在 widget 创建时固定，重建即可：

```python
vt.close()
vt = faultviz.ViewWidget(live_rows=1000, stats_bucket=10, columns=COLUMNS)
vt.show(replace=True)
```

其他可调（不填都有合理默认）：

- `stats_bucket`（桶宽/最小单位，默认 100；`0`=关面板）；
- 横纵轴语义：`stats_state`（分类列名，默认 `state`）、`stats_delay`（分桶列名，默认 `delay`）、
  `stats_success`（成功状态值，默认 `PASS`）、`stats_hang`（HANG 状态值，默认 `HANG`）；
- `stats_custom`：自定义第三状态值（用法同上两个，图表占右侧整列；默认 `""`=不显示）；
- 柱数 `stats_bars`（默认 30，最多 50）、推送节流 `stats_interval`（默认 5 帧一次）。